In [33]:
import pandas as pd
from lifelines import KaplanMeierFitter
df = pd.read_csv("masked_data_V2_processed.csv")
df.columns = df.columns.str.strip()
print(df.columns.tolist())

['series number', 'product type', 'model', 'repair operation number', 'previous repair operation number', 'time since previous repair', 'censored', 'age after repair', 'repair center', 'Component A', 'Component B', 'Component C', 'Component D', 'Component E', 'Component F', 'Component H', 'Component I', 'Component J', 'Component K', 'next_failure_time', 'next_failure_time_censored']


In [34]:
t_col = "time since previous repair"
event_col = "censored"
df["event_observed"] = 1 - df[event_col]
models = ["V1E1", "V1E2", "V1E3", "V2E2", "V2E3"]
km = KaplanMeierFitter()
r500 = {}
for m in models:
    sub = df[df["model"] == m].dropna(subset=[t_col, "event_observed"])
    km.fit(sub[t_col], event_observed=sub["event_observed"], label=m)
    r500[m] = float(km.predict(500))
pd.Series(r500, name="R500_hat").sort_values(ascending=False)


V2E2    0.971941
V2E3    0.953576
V1E3    0.934753
V1E1    0.921687
V1E2    0.873552
Name: R500_hat, dtype: float64

In [35]:
import numpy as np
def km_r500(d, e, t=500):
    km = KaplanMeierFitter().fit(d, e)
    return float(km.survival_function_at_times(t).iloc[0])
B = 300
t = 500
out = []
rng = np.random.default_rng(0)
for m in models:
    sub = df[df["model"]==m].dropna(subset=[t_col, "event_observed"])
    d = sub[t_col].to_numpy()
    e = sub["event_observed"].to_numpy()
    n = len(sub)
    boots = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        boots.append(km_r500(d[idx], e[idx], t=t))
    r = km_r500(d, e, t=t)
    lo, hi = np.quantile(boots, [0.025, 0.975])
    out.append({"model": m, "n": n, "R500": r, "CI_low": lo, "CI_high": hi})
pd.DataFrame(out).sort_values("R500", ascending=False)

,model,n,R500,CI_low,CI_high
3,V2E2,3646,0.971941,0.967103,0.976409
4,V2E3,2468,0.953576,0.944918,0.961865
2,V1E3,725,0.934753,0.917074,0.953989
0,V1E1,2056,0.921687,0.909244,0.932513
1,V1E2,3925,0.873552,0.862066,0.884009


In [36]:
t_col = "time since previous repair"
bias_tbl = (df.groupby("model")
              .agg(
                  n=("model","size"),
                  censor_rate=("censored","mean"),
                  median_time=(t_col,"median"),
                  mean_time=(t_col,"mean"),
                  mean_age=("age after repair","mean"),
                  median_age=("age after repair","median"),
                  n_centers=("repair center","nunique"),
              )
           ).sort_values("n", ascending=False)
bias_tbl

,n,censor_rate,median_time,mean_time,mean_age,median_age,n_centers
model,,,,,,,
V1E2,3925,0.466497,1199.0,1277.311083,2255.899618,2439.0,6
V2E2,3646,0.679923,1430.0,1444.087767,1560.882611,1524.0,5
V2E3,2468,0.939627,652.0,636.185575,1140.557131,944.0,4
V1E1,2056,0.242704,1246.5,1585.221790,1918.486868,1615.5,5
V1E3,725,0.937931,504.0,494.532414,3189.619310,3072.0,2


In [37]:
import numpy as np
t = 500
t_col = "time since previous repair"
df["event_observed"] = 1 - df["censored"]
df["Y_fail_before_500"] = np.nan
df.loc[(df["event_observed"]==1) & (df[t_col] <= t), "Y_fail_before_500"] = 1
df.loc[df[t_col] >= t, "Y_fail_before_500"] = 0
work = df.dropna(subset=["Y_fail_before_500"]).copy()

In [38]:
from lifelines import KaplanMeierFitter
km_c = KaplanMeierFitter()
km_c.fit(df[t_col], event_observed=df["censored"]) 
u = np.minimum(work[t_col].to_numpy(), t)  # min(time, 500)
Ghat = km_c.survival_function_at_times(u).to_numpy()
eps = 1e-6
work["ipcw"] = 1.0 / np.clip(Ghat, eps, None)

In [39]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
T = "model"
W = ["product type", "repair center", "age after repair", "repair operation number"]
X_cols = [T] + W
y = work["Y_fail_before_500"].astype(int)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"),
     [c for c in X_cols if work[c].dtype == "object"]),
    ("num", "passthrough",
     [c for c in X_cols if work[c].dtype != "object"]),
])
clf = Pipeline([
    ("pre", pre),
    ("lr", LogisticRegression(max_iter=2000))
])
clf.fit(work[X_cols], y, lr__sample_weight=work["ipcw"])
models = ["V1E1", "V1E2", "V1E3", "V2E2", "V2E3"]
R500_causal = {}
for m in models:
    tmp = work[X_cols].copy()
    tmp["model"] = m
    EY = clf.predict_proba(tmp)[:, 1].mean()   
    R500_causal[m] = 1 - EY                   
pd.Series(R500_causal, name="R500_causal").sort_values(ascending=False)

V2E3    0.962121
V1E1    0.950123
V2E2    0.943638
V1E2    0.873360
V1E3    0.637538
Name: R500_causal, dtype: float64

In [43]:
import pandas as pd
out_df = pd.DataFrame(out).sort_values("R500", ascending=False)
emp = (out_df.rename(columns={
            "R500": "R500_emp",
            "CI_low": "R500_emp_CI_low",
            "CI_high": "R500_emp_CI_high",
            "n": "n_emp"  
        })
        .set_index("model")
     )
bias = bias_tbl.copy()
bias = bias.rename(columns={"n": "n_total"})  
causal = pd.Series(R500_causal, name="R500_causal").to_frame()
summary = emp.join(bias, how="left").join(causal, how="left")
summary["delta_causal_minus_emp"] = summary["R500_causal"] - summary["R500_emp"]
summary = summary.reset_index()
summary = summary[[
    "model",
    "n_emp", "n_total", "censor_rate", "n_centers",
    "R500_emp", "R500_emp_CI_low", "R500_emp_CI_high",
    "R500_causal", "delta_causal_minus_emp",
    "mean_age", "median_age", "median_time", "mean_time",
]]
summary.sort_values("R500_emp", ascending=False)

,model,n_emp,n_total,censor_rate,n_centers,R500_emp,R500_emp_CI_low,R500_emp_CI_high,R500_causal,delta_causal_minus_emp,mean_age,median_age,median_time,mean_time
0,V2E2,3646,3646,0.679923,5,0.971941,0.967103,0.976409,0.943638,-0.028303,1560.882611,1524.0,1430.0,1444.087767
1,V2E3,2468,2468,0.939627,4,0.953576,0.944918,0.961865,0.962121,0.008545,1140.557131,944.0,652.0,636.185575
2,V1E3,725,725,0.937931,2,0.934753,0.917074,0.953989,0.637538,-0.297214,3189.619310,3072.0,504.0,494.532414
3,V1E1,2056,2056,0.242704,5,0.921687,0.909244,0.932513,0.950123,0.028436,1918.486868,1615.5,1246.5,1585.221790
4,V1E2,3925,3925,0.466497,6,0.873552,0.862066,0.884009,0.873360,-0.000192,2255.899618,2439.0,1199.0,1277.311083


# Task 2 

In [41]:
import numpy as np
import pandas as pd
df = pd.read_csv("masked_data_V2_processed.csv")
df.columns = df.columns.str.strip()  
t = 500
t_col = "time since previous repair"
cens_col = "censored"
center_col = "repair center"
df["event_observed"] = 1 - df[cens_col]
centers = sorted(df[center_col].dropna().unique())
print("Centers:", centers)
df[[t_col, cens_col, "event_observed", center_col]].head()

Centers: ['center_0', 'center_1', 'center_2', 'center_3', 'center_4', 'center_5', 'center_6']


,time since previous repair,censored,event_observed,repair center
0,1260,0,1,center_0
1,1319,0,1,center_1
2,969,1,0,center_1
3,2159,0,1,center_0
4,1410,1,0,center_2


In [44]:
from lifelines import KaplanMeierFitter
km = KaplanMeierFitter()
rows = []
for c in centers:
    sub = df[df[center_col] == c].dropna(subset=[t_col, "event_observed"])
    km.fit(sub[t_col], event_observed=sub["event_observed"], label=str(c))
    r500 = float(km.survival_function_at_times(t).iloc[0])
    rows.append({"repair_center": c, "n": len(sub), "R500_emp_KM": r500})
emp_center_tbl = pd.DataFrame(rows).sort_values("R500_emp_KM", ascending=False)
emp_center_tbl

,repair_center,n,R500_emp_KM
4,center_4,20,1.000000
5,center_5,2,1.000000
6,center_6,1,1.000000
0,center_0,7331,0.947078
3,center_3,93,0.946237
2,center_2,4829,0.899261
1,center_1,544,0.883118


In [50]:
from lifelines import KaplanMeierFitter
def km_r500(d, e, t=500):
    km = KaplanMeierFitter().fit(d, e)
    return float(km.survival_function_at_times(t).iloc[0])
B = 300
rng = np.random.default_rng(0)
rows = []
for c in centers:
    sub = df[df[center_col] == c].dropna(subset=[t_col, "event_observed"])
    d = sub[t_col].to_numpy()
    e = sub["event_observed"].to_numpy()
    n = len(sub)
    boots = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        boots.append(km_r500(d[idx], e[idx], t=t))
    r = km_r500(d, e, t=t)
    lo, hi = np.quantile(boots, [0.025, 0.975])
    rows.append({"repair_center": c, "n": n, "R500_emp": r, "CI_low": lo, "CI_high": hi})
emp_center_ci_tbl = pd.DataFrame(rows).sort_values("R500_emp", ascending=False)
emp_center_ci_tbl

,repair_center,n,R500_emp,CI_low,CI_high
4,center_4,20,1.000000,1.000000,1.000000
5,center_5,2,1.000000,1.000000,1.000000
6,center_6,1,1.000000,1.000000,1.000000
0,center_0,7331,0.947078,0.941341,0.951608
3,center_3,93,0.946237,0.892473,0.984140
2,center_2,4829,0.899261,0.890743,0.908810
1,center_1,544,0.883118,0.854870,0.907554


In [51]:
bias_center_tbl = (df.groupby(center_col)
    .agg(
        n=(center_col, "size"),
        censor_rate=(cens_col, "mean"),
        median_time=(t_col, "median"),
        mean_time=(t_col, "mean"),
        mean_age=("age after repair", "mean"),
        median_age=("age after repair", "median"),
        model_nunique=("model", "nunique"),
        product_type_nunique=("product type", "nunique"),
    )
    .sort_values("n", ascending=False)
)
bias_center_tbl

,n,censor_rate,median_time,mean_time,mean_age,median_age,model_nunique,product_type_nunique
repair center,,,,,,,,
center_0,7331,0.586687,1272.0,1383.845587,1396.794025,1279.0,4,2
center_2,4829,0.636778,821.0,915.402361,2423.024850,2378.0,5,2
center_1,544,0.608456,1192.5,1256.996324,2525.827206,2485.0,5,2
center_3,93,0.838710,2382.0,2239.440860,2710.354839,2611.0,3,2
center_4,20,1.000000,80.0,281.300000,2075.050000,1785.5,3,2
center_5,2,1.000000,1003.0,1003.000000,2517.500000,2517.5,1,1
center_6,1,1.000000,1292.0,1292.000000,3829.000000,3829.0,1,1


In [46]:
pd.crosstab(df[center_col], df["model"], normalize="index")

model,V1E1,V1E2,V1E3,V2E2,V2E3
repair center,,,,,
center_0,0.203246,0.160551,0.000000,0.428045,0.208157
center_1,0.071691,0.753676,0.012868,0.154412,0.007353
center_2,0.103748,0.473390,0.148685,0.080141,0.194036
center_3,0.268817,0.494624,0.000000,0.236559,0.000000
center_4,0.000000,0.200000,0.000000,0.750000,0.050000
center_5,0.000000,1.000000,0.000000,0.000000,0.000000
center_6,1.000000,0.000000,0.000000,0.000000,0.000000


In [47]:
df["Y_fail_before_500"] = np.nan
fail = (df["event_observed"] == 1)
df.loc[fail & (df[t_col] <= t), "Y_fail_before_500"] = 1
df.loc[df[t_col] >= t, "Y_fail_before_500"] = 0
work = df.dropna(subset=["Y_fail_before_500"]).copy()
work["Y_fail_before_500"] = work["Y_fail_before_500"].astype(int)
work[[center_col, t_col, cens_col, "Y_fail_before_500"]].head()

,repair center,time since previous repair,censored,Y_fail_before_500
0,center_0,1260,0,0
1,center_1,1319,0,0
2,center_1,969,1,0
3,center_0,2159,0,0
4,center_2,1410,1,0


In [48]:
from lifelines import KaplanMeierFitter
eps = 1e-6
work["ipcw"] = np.nan
km_c = KaplanMeierFitter()
for c in centers:
    df_c = df[df[center_col] == c].dropna(subset=[t_col, cens_col])
    km_c.fit(df_c[t_col], event_observed=df_c[cens_col])  
    idx = work[center_col] == c
    u = np.minimum(work.loc[idx, t_col].to_numpy(), t)
    Ghat = km_c.survival_function_at_times(u).to_numpy()
    work.loc[idx, "ipcw"] = 1.0 / np.clip(Ghat, eps, None)
work.groupby(center_col)["ipcw"].describe(percentiles=[.5, .9, .95, .99])

,count,mean,std,min,50%,90%,95%,99%,max
repair center,,,,,,,,,
center_0,7084.0,1.034867,6.207297e-03,1.000000,1.036268,1.036268,1.036268,1.036268,1.036268
center_1,537.0,1.013035,2.009168e-03,1.001842,1.013536,1.013536,1.013536,1.013536,1.013536
center_2,3889.0,1.241707,5.053376e-02,1.000000,1.257185,1.257185,1.257185,1.257185,1.257185
center_3,93.0,1.000000,0.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
center_4,3.0,6.666667,1.087792e-15,6.666667,6.666667,6.666667,6.666667,6.666667,6.666667
center_5,2.0,1.000000,0.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
center_6,1.0,1.000000,NaN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [49]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
T = center_col
W = ["model", "product type", "age after repair", "repair operation number"]
X_cols = [T] + W
y = work["Y_fail_before_500"]
pre = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         [c for c in X_cols if work[c].dtype == "object"]),
        ("num", "passthrough",
         [c for c in X_cols if work[c].dtype != "object"]),
    ]
)
clf = Pipeline(steps=[
    ("pre", pre),
    ("lr", LogisticRegression(max_iter=2000))
])
clf.fit(work[X_cols], y, lr__sample_weight=work["ipcw"])

R500_causal_center = {}
for c in centers:
    tmp = work[X_cols].copy()
    tmp[center_col] = c
    EY = clf.predict_proba(tmp)[:, 1].mean() 
    R500_causal_center[c] = 1 - EY
causal_center_tbl = pd.Series(R500_causal_center, name="R500_causal").sort_values(ascending=False)
causal_center_tbl

center_0    0.961325
center_4    0.902859
center_5    0.883040
center_6    0.882437
center_1    0.827474
center_3    0.802956
center_2    0.776752
Name: R500_causal, dtype: float64

In [32]:
emp = emp_center_ci_tbl.set_index("repair_center")[["n", "R500_emp", "CI_low", "CI_high"]]
causal = causal_center_tbl.to_frame()
task2_summary = emp.join(causal, how="left")
task2_summary["delta_causal_minus_emp"] = task2_summary["R500_causal"] - task2_summary["R500_emp"]
task2_summary.sort_values("R500_emp", ascending=False)

,n,R500_emp,CI_low,CI_high,R500_causal,delta_causal_minus_emp
repair_center,,,,,,
center_4,20,1.000000,1.000000,1.000000,0.902859,-0.097141
center_5,2,1.000000,1.000000,1.000000,0.883040,-0.116960
center_6,1,1.000000,1.000000,1.000000,0.882437,-0.117563
center_0,7331,0.947078,0.941341,0.951608,0.961325,0.014246
center_3,93,0.946237,0.892473,0.984140,0.802956,-0.143281
center_2,4829,0.899261,0.890743,0.908810,0.776752,-0.122509
center_1,544,0.883118,0.854870,0.907554,0.827474,-0.055643
